# Install PyMongo

In [ ]:
!pip install pymongo

# EXECUTE FIRST!

In [ ]:

from pymongo import MongoClient
uri = "mongodb://localhost:27017/"
def execute(callable):
    client = MongoClient(uri)
    try:
        callable(client)

        client.close()
    except Exception as e:
        raise Exception("Unable to find the document due to the following error: ", e)

# Example 1 : Access collections

In [ ]:
def access_collections(client):
    db = client['classic']
    customers = db.customers # collection customers
    orders = db.orders # collection orders
    products = db.products # collection products
    orderdetails = db.orderdetails # collection orderdetails

    print(customers)
    print(orders)
    print(products)
    print(orderdetails)

execute(access_collections)

# Example 2 : Find offices only in San Francisco

In [ ]:
def find_offices(client):
    database = client.get_database("classic")
    offices_collection = database.get_collection("offices")
    query = {"city": "San Francisco"}
    offices_cursor = offices_collection.find(query)
    office_lists = offices_cursor.to_list()
    for i in range(len(office_lists)):
        print(office_lists[i])

execute(find_offices)

# Example 3 : Navigate document

In [ ]:
customer_doc = {
    "customerNumber": 103,
    "name": "Atelier graphique",
    "contact": {
        "firstName": "Carine",
        "lastName": "Schmitt",
        "phone": "40.32.2555"
    },
    "address": {
        "street": "54, rue Royale",
        "city": "Nantes", 
        "country": "France",
        "postalCode": "44000"
    },
    "creditLimit": 21000,
    "salesRep": {
        "employeeNumber": 1370,
        "name": "Gerard Hernandez"
    }
}

# Address isn't a separate table - it's part of the customer
customer_city = customer_doc["address"]["city"]          # Easy navigation
customer_name = customer_doc["contact"]["firstName"]     # Intuitive structure

print(customer_city, customer_name)

# Example 4 : Find documents

In [ ]:
def find_customers_contact(client):

    database = client['classic']
    customers = database["customers"]

    # Find customers who HAVE a contact person
    customers_with_sales = customers.find({
        "salesRepEmployeeNumber": {"$exists": True}
    })

    # Find customers who DON'T have a contact person
    customers_without_sales = customers.find({
        "salesRepEmployeeNumber": {"$exists": False}
    })

    for customer in customers.find():
        name = customer.get("contactFirstName")
        contact = customer.get("salesRepEmployeeNumber", {}) # Default to empty dict

        if contact:
            print(f"{name} - Contact : {contact}")
        else:
            print(f"{name} - No individual contact")

execute(find_customers_contact)

# Example 5 : Insert documents

In [ ]:
def insert_customers(client):

    database = client['classic']
    customers = database["customers2"]


    # Insert a customer (collection created automatically)
    customer = {
        "customerNumber": 103,
        "name": "Atelier graphique",
        "contact": {
            "firstName": "Carine",
            "lastName": "Schmitt",
            "phone": "40.32.2555"
        },
        "address": {
            "street": "54, rue Royale",
            "city": "Nantes",
            "country": "France", 
            "postalCode": "44000"
        },
        "creditLimit": 21000
    }
    
    # Insert one customer
    result = customers.insert_one(customer)
    print(f"Inserted customer with ID: {result.inserted_id}")
    
    # Insert multiple customers
    customer_list = [
        {
            "customerNumber": 112,
            "name": "Signal Gift Stores",
            "address": {
                "street": "8489 Strong St.",
                "city": "Las Vegas",
                "country": "USA",
                "postalCode": "83030"
            },
            "creditLimit": 71800
        },
        {
            "customerNumber": 114, 
            "name": "Australian Collectors, Co.",
            "contact": {
                "firstName": "Peter",
                "lastName": "Ferguson"
            },
            "address": {
                "street": "636 St Kilda Road",
                "city": "Melbourne",
                "country": "Australia", 
                "postalCode": "3004"
            },
            "creditLimit": 117300
        }
    ]
    
    result = customers.insert_many(customer_list)
    print(f"Inserted {len(result.inserted_ids)} customers")

def drop_customers(client):
    db = client['classic']
    customers2 = db['customers2']
    customers2.drop()

execute(drop_customers)
execute(insert_customers)

# Example 6: Find documents with query

In [ ]:
def find_document(client):
    database = client['classic']
    customers = database["customers2"]

    for customer in customers.find({"contact.firstName":"Peter","address.country":"Australia","creditLimit":{"$gt":5000}}):
        print(customer)

execute(find_document)

In [ ]:
def find_customers(client):
    database = client['classic']
    customers = database['customers2']

    for customer in customers.find({'address.country':'USA'}):
        print(customer)

execute(find_customers)

# Example 7: Projection Selecting Fields

In [ ]:
def show_customer_name_country(client):
    database = client['classic']
    customers = database['customers2']

    for customer in customers.find({"address.country": "USA"},
    {"name": 1, "address.country": 1, "_id": 0}):
        print(customer)

execute(show_customer_name_country)

# Example 8: Exclude fields

In [ ]:
def show_customer_name_creditlimit(client):
    database = client['classic']
    customers = database['customers2']


    # Exclude specific fields
    no_contact = customers.find({}, {"contact": 0, "_id": 0, "address":0})

    for customer in no_contact:
        print(customer)

execute(show_customer_name_country)

In [ ]:
def selecting_fields(client):
    database = client['classic']
    customers = database['customers2']


    # Create a clean display
    usa_customers = customers.find(
        {"address.country": "USA"},
        { "name": 1, "city": "$address.city", "creditLimit": 1, "_id": 0 }
    )
    for customer in usa_customers:
        print(customer)

execute(selecting_fields)

In [ ]:
def selecting_fields2(client):
    database = client['classic']
    customers = database['customers2']


    # Create a clean display
    usa_customers = customers.find(
        {"address.country": "USA"},
        { "name": 1, "city": "$address.city", "creditLimit": 1, "_id": 0 }
    )
    print(type(usa_customers))
    usa_list = list(usa_customers)
    for customer in usa_list:
        print(f"{customer['name']} - Credit: ${customer['creditLimit']:,}")

execute(selecting_fields2)

# Example 9: Insert products

In [ ]:
def insert_products(client):
    db = client['classic']
    products = db['products2']
    products.drop()
    # Insert products with rich document structure
    product_list = [
        {
            "productCode": "S18_1749",
            "name": "1917 Grand Touring Sedan", 
            "description": "This 1:18 scale replica of the 1917 Grand Touring car...",
            "productLine": "Vintage Cars",
            "scale": "1:18",
            "vendor": "Welly Diecast Productions",
            "quantityInStock": 2724,
            "pricing": {
                "buyPrice": 86.70,
                "MSRP": 170.00,
                "margin": 83.30
            },
            "specifications": {
                "weight": "1.5 lbs",
                "dimensions": {
                    "length": "10 inches",
                    "width": "4 inches", 
                    "height": "3 inches"
                }
            }
        },
        {
        "productCode": "S18_2248",
        "name": "1911 Ford Town Car",
        "description": "Features opening hood, opening doors, opening trunk...",
        "productLine": "Vintage Cars", 
        "scale": "1:18",
        "vendor": "Motor City Art Classics",
        "quantityInStock": 540,
        "pricing": {
            "buyPrice": 33.30,
            "MSRP": 60.54,
            "margin": 27.24
        }
    }
    ]

    # Insert products
    result = products.insert_many(product_list)
    print(f"Inserted {len(result.inserted_ids)} products")

execute(insert_products)

# Example 10

In [ ]:
def vintage_cars(client):
    db = client['classic']
    products = db['products2']
    # Find vintage cars
    vintage_cars = products.find({"productLine": "Vintage Cars"})
    print("Vintage Cars:")
    for car in vintage_cars:
        print(f"- {car['name']} (Stock: {car['quantityInStock']})")

    # Find products with low stock
    low_stock = products.count_documents({"quantityInStock": {"$lt": 1000}})
    print(f"\nLow stock products: {low_stock}")

    # Find by price range
    affordable_products = products.find({
        "pricing.MSRP": {"$gte": 50, "$lte": 100}
    })
    print("\nAffordable products ($50-$100):")
    for product in affordable_products:
        price = product['pricing']['MSRP']
        print(f"- {product['name']}: ${price}")


execute(vintage_cars)

In [ ]:
def customer_range(client):
    db = client['classic']
    customers = db['customers2']
    customers2 = customers.find({'address.country':{'$in':['USA','France']},'$and':[{'creditLimit':{'$gt':30000}},{'creditLimit':{'$lt':80000}}]})
    for customer in customers2:
        print(customer)

    # Field exists
    has_contact = customers.find({"contact.firstName": {"$exists": True}})
    for customer in has_contact:
        print(f"Contact - {customer['contact']['firstName']}")

    # Pattern matching (regex)
    query = {"name": {"$regex": "Gift", "$options": "i"}}
    gift_stores = customers.find(query)
    count_gift_stores = customers.count_documents(query)  # Case insensitive

    # Print results with count
    print(f"Gift stores found: {count_gift_stores}")
    for store in gift_stores:
        print(f"- {store['name']}")

execute(customer_range)

# Example 11: Sorting

In [ ]:
def sort_customer(client):
    db = client['classic']
    customers = db['customers2']
    # Sort by credit limit (ascending)
    customers_asc = customers.find().sort("creditLimit", 1)
    print("Customers by credit limit (ascending):")
    for customer in customers_asc.limit(3):
        print(f"- {customer['name']}: ${customer['creditLimit']:,}")

    # Sort by credit limit (descending)  
    customers_desc = customers.find().sort("creditLimit", -1)
    print("\nTop 3 customers by credit limit:")
    for customer in customers_desc.limit(3):
        print(f"- {customer['name']}: ${customer['creditLimit']:,}")


    # Sort by multiple fields
    multi_sort = customers.find().sort([
        ("address.country", 1),
        ("creditLimit", -1)
    ])

    print("\nSorting by multiple fields")
    for customer in multi_sort:
        print(f"- {customer['name']}: ${customer['creditLimit']:,} lives in {customer['address']['country']}")


    # Skip and limit (pagination)
    skip = 1
    limit = 3
    page_2 = customers.find().skip(skip).limit(limit)
    print(f"\nPage 2 customers ({skip}-{limit}):")
    for i, customer in enumerate(page_2, skip):
        print(f"{i}. {customer['name']}")

    # Combine operations with method chaining
    usa_top_customers = customers.find({"address.country": "USA"}) \
                                .sort("creditLimit", -1) \
                                .limit(3)
    
    print("\nTop 3 USA customers:")
    for customer in usa_top_customers:
        print(f"- {customer['name']}: ${customer['creditLimit']:,}")

execute(sort_customer)

# Aggregation

# Example 12: match

In [ ]:
def match_usa_customers(client):
    db = client['classic']
    customers = db['customers']
    pipeline = [
        {
            "$match": {
                "country": "USA"
            }
        }
    ]
    usa_customer = customers.aggregate(pipeline)
    for customer in usa_customer:
        print(f"Country: {customer['country']}, name: {customer['customerName']}")

execute(match_usa_customers)

# Example 13: projects

In [ ]:
def basic_projection(client):
    db = client['classic']
    customers = db['customers']
    basic_projection = {"$project": {"name": "$customerName", "creditLimit": 1, "_id": 0}}
    pipeline = [basic_projection]
    credit_customers = customers.aggregate(pipeline)
    for customer in credit_customers:
        print(f"customer: {customer['name']} - creditLimit: {customer['creditLimit']}")

execute(basic_projection)

# Example 14: group and sum

In [ ]:
def count_customer_by_country(client):
    db = client['classic']
    customers = db['customers']
    # Simple aggregation: count customers by country
    pipeline = [
        {
            "$group": {
                "_id": "$country",           # Group by country
                "customerCount": {"$sum": 1},         # Count documents in each group                
            },
        }
    ]

    country_counts = customers.aggregate(pipeline)
    print(f"Customers by country: ")
    for result in country_counts:
        print(f"- {result['_id']}: {result['customerCount']} customers")



    # Same thing with more Pythonic approach
    country_pipeline = [
        {"$group": {"_id": "$country", "count": {"$sum": 1}}},
        {"$sort": {"count": -1}}  # Sort by count descending
    ]

    results = list(customers.aggregate(country_pipeline))
    for result in results:
        country = result['_id'] or 'Unknown'
        count = result['count']
        print(f"{country}: {count} customers")

execute(count_customer_by_country)

# Example 15 : Find one

In [ ]:
def find_one_customer(client):
    db = client['classic']
    customers = db['customers']
    customer = customers.find_one()
    print(customer)

execute(find_one_customer)